<div style="
    background: linear-gradient(to right,rgb(246, 37, 156),rgb(175, 56, 244));
    padding: 5px;
    border-radius: 8px;
    box-shadow: 2px 2px 8px rgb(255, 255, 255);
">
  <h3 style="
      color: Black;
      font-size: 18px;
      font-weight: 800;
      font-family: 'Segoe UI', sans-serif;
      margin: 0;
  ">Deployment: Dashboard</h3>
</div>

**Smarter Campaign Targeting DashBoard**
- Feature importance: Explore feature influence
- Single client prediction tab: predict term deposit subscriptions
- Batch CSV prediction tab: run batch or single client predictions

**Importing Libraries**

In [1]:
import gradio as gr
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

# Load model and feature list
model = joblib.load("xgboost_model.pkl")
input_features = joblib.load("input_features.pkl")  # should include all 25 features
full_feature_names = model.get_booster().feature_names

#input fields required from the user (raw inputs or that map to encoded one-hot)
input_fields = [
    "yearly_balance", "age", "last_contact_date", "duration", "previous_days", "this_campaign",  # numeric
    "previous_outcome_success", "housing_loan_yes", "contact_type_unknown",                     # binary
    "last_contact_mth", "marital_status", "education_level"                                     # dropdowns (one-hot)
]

#descriptions for users inputs
feature_descriptions = {
    "yearly_balance": "Client's yearly balance (EUR)",
    "age": "Client's age",
    "last_contact_date": "Date of last contact (numeric or categorical as processed)",
    "duration": "Duration of last contact in seconds",
    "previous_days": "Days since last contact",
    "this_campaign": "Number of contacts in this campaign",
    "previous_outcome_success": "1 = client subscribed previously, else 0",
    "housing_loan_yes": "1 = has housing loan, else 0",
    "contact_type_unknown": "1 = contact type unknown, else 0",
    "last_contact_mth": "Month of last contact (Jan–Dec)",
    "marital_status": "Client's marital status",
    "education_level": "Client's education level"
}

<div style="
    background: linear-gradient(to right,rgb(246, 37, 156),rgb(175, 56, 244));
    padding: 5px;
    border-radius: 8px;
    box-shadow: 2px 2px 8px rgb(255, 255, 255);
">
  <h3 style="
      color: Black;
      font-size: 18px;
      font-weight: 800;
      font-family: 'Segoe UI', sans-serif;
      margin: 0;
  ">Feature Importance Tab</h3>
</div>

In [2]:
#feature importance plotting function
def plot_feature_importance():
    importances = model.feature_importances_
    feat_series = pd.Series(importances, index=full_feature_names).sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(8, 6))
    feat_series.head(20).plot(kind="barh", ax=ax, color="#fa1fcf")
    ax.set_title("Top 20 Feature Importances")
    ax.invert_yaxis()
    return fig

#interface
feature_tab = gr.Interface(
    fn=plot_feature_importance,
    inputs=[],
    outputs="plot",
    title="Feature Importance",
    description=(
        "Understand which customer features most influence term deposit subscriptions.\n"
        "\nClick Generate. This can help prioritize high-impact client attributes like balance, duration, or past outcomes.\n"))

<div style="
    background: linear-gradient(to right,rgb(246, 37, 156),rgb(175, 56, 244));
    padding: 5px;
    border-radius: 8px;
    box-shadow: 2px 2px 8px rgb(255, 255, 255);
">
  <h3 style="
      color: Black;
      font-size: 18px;
      font-weight: 800;
      font-family: 'Segoe UI', sans-serif;
      margin: 0;
  ">Single Client Prediction Tab</h3>
</div>

In [3]:
#defining prediction single function
def predict_single(
    yearly_balance, age, last_contact_date, duration, previous_days, this_campaign,
    previous_outcome_success, housing_loan_yes,
    contact_type_unknown,
    last_contact_mth, marital_status, education_level):
    
   #prepare a zero-filled df with all expected model features, just in case user did not fill up the fields
    base = pd.DataFrame([np.zeros(len(full_feature_names))], columns=full_feature_names)

    #assigning numerical and binary values
    base["yearly_balance"] = yearly_balance
    base["age"] = age
    base["last_contact_date"] = last_contact_date
    base["duration"] = duration
    base["previous_days"] = previous_days
    base["this_campaign"] = this_campaign
    base["previous_outcome_success"] = previous_outcome_success
    base["housing_loan_yes"] = housing_loan_yes
    base["contact_type_unknown"] = contact_type_unknown

    #dropdowns for user to avoid wrong entries especially to encoded columns e.g last_contact_mth_???
    mth_col = f"last_contact_mth_{last_contact_mth}"
    if mth_col in base.columns:
        base[mth_col] = 1

    marital_col = f"marital_status_{marital_status}"
    if marital_col in base.columns:
        base[marital_col] = 1

    edu_col = f"education_level_{education_level}"
    if edu_col in base.columns:
        base[edu_col] = 1

    #prediction
    pred = model.predict(base)[0]
    prob = model.predict_proba(base)[0][1]
    return ("yes" if pred == 1 else "no"), f"{prob:.2%}" #easier for users to read and in %

#input fields for user for easier input
input_widgets = [
    gr.Number(label="Yearly Balance (EUR)"),
    gr.Number(label="Age"),
    gr.Number(label="Last Contact Day of Month (e.g. 15 for 15th)"),
    gr.Number(label="Call Duration (Seconds)"),
    gr.Number(label="Days Since Last Contact"),
    gr.Number(label="Number of Contacts (This Campaign)"),
    gr.Radio(choices=[0, 1], label="Previously Subscribed (1 = Yes, 0 = No)"),
    gr.Radio(choices=[0, 1], label="Housing Loan (1 = Yes, 0 = No)"),
    gr.Radio(choices=[0, 1], label="Contact Type Unknown (1 = Yes, 0 = No)"),
    gr.Dropdown(choices=[
        "jan", "feb", "mar", "apr", "may", "jun",
        "jul", "aug", "sep", "oct", "nov", "dec"
    ], label="Last Contact Month"),
    gr.Dropdown(choices=["married", "single"], label="Marital Status"),
    gr.Dropdown(choices=["secondary", "tertiary", "unknown"], label="Education Level")]

#interface
single_tab = gr.Interface(
    fn=predict_single,
    inputs=input_widgets,
    outputs=[
        gr.Textbox(label="Prediction (yes = Will Subscribe, no = Will Not Subscribe)"),
        gr.Textbox(label="Probability of Subscription")],

    title="Single Client Prediction",
    description=(
        "Enter client details to predict whether they'll subscribe to a term deposit.\n\n"
        "- Use 1 = Yes, 0 = No for binary questions.\n"
        "- Dropdowns auto-map to encoded features.\n"
        "You’ll receive both the prediction and the confidence score."))

<div style="
    background: linear-gradient(to right,rgb(246, 37, 156),rgb(175, 56, 244));
    padding: 5px;
    border-radius: 8px;
    box-shadow: 2px 2px 8px rgb(255, 255, 255);
">
  <h3 style="
      color: Black;
      font-size: 18px;
      font-weight: 800;
      font-family: 'Segoe UI', sans-serif;
      margin: 0;
  ">Batch Upload Tab</h3>
</div>

In [4]:
#upload predict file
def predict_batch(file):
    df = pd.read_csv(file.name)

    #prepare a zero-filled df with all expected model features, just in case user did not fill up the fields
    X = pd.DataFrame(np.zeros((df.shape[0], len(full_feature_names))), columns=full_feature_names)

    #looping only mapped features
    for col in df.columns:
        if col in X.columns:
            X[col] = df[col]

    # Generate predictions
    preds = pd.Series(model.predict(X).astype(int)) # ensures the result is clean numerical
    df["Prediction"] = preds.map({0: "no", 1: "yes"})
    df["Probability (yes)"] = model.predict_proba(X)[:, 1]

    #save and return output results
    output_path = "predicted_output.csv"
    df.to_csv(output_path, index=False)
    return df, output_path

In [5]:
#interface part
with gr.Blocks() as batch_tab:
    gr.Markdown("## Batch Upload — Predict in Bulk")
    gr.Markdown(
        "**Instructions:**\n" #instructions for users
        "1. Download the blank CSV template below.\n"
        "2. Fill it with customer data. Do not rename or remove any column headers.\n"
        "3. Upload the completed CSV file.\n"
        "4. Click **Predict** to get predictions and probability scores for each entry."
    )

    #downloadable template CSV file
    gr.File(value="blank_template.csv", label="Download CSV Template", interactive=False)

    #upload field and prediction interface
    upload_file = gr.File(file_types=[".csv"], label="Upload CSV File")
    predict_btn = gr.Button("Predict")
    results_table = gr.Dataframe()
    download_file = gr.File(label="Download Prediction CSV")

    #link prediction function output of results
    predict_btn.click(fn=predict_batch, inputs=upload_file, outputs=[results_table, download_file])

<div style="
    background: linear-gradient(to right,rgb(246, 37, 156),rgb(175, 56, 244));
    padding: 5px;
    border-radius: 8px;
    box-shadow: 2px 2px 8px rgb(255, 255, 255);
">
  <h3 style="
      color: Black;
      font-size: 18px;
      font-weight: 800;
      font-family: 'Segoe UI', sans-serif;
      margin: 0;
  ">Interface Header</h3>
</div>

In [6]:
#header 
with gr.Blocks() as dashboard:

    gr.Markdown(
        """
        # Smarter Campaign Targeting (Dashboard)  
        # Cohort 8 Banking 
        <div style='font-size: 16px; color: #666; margin-top: -10px;'>A Smarter Way of Banking</div>
        """,
        elem_id="dashboard"
    )

    gr.TabbedInterface(
        [feature_tab, single_tab, batch_tab],
        ["Feature Importance", "Single Prediction", "Batch Upload"] #tab name
    )

dashboard.launch(inline=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
